In [34]:
using Base
ENV["JULIA_NUM_THREADS"] = "auto"
Threads.nthreads()

1

In [ ]:

using Pkg

cd(@__DIR__)

Pkg.activate("../")
ParamFile = "../config/testparam.csv" # 1D Earth/Mars/Moon models are defined

include("../src/commonBatchs.jl")
include("../src/planet1D.jl")
include("../src/GeoPoints.jl")

using .commonBatchs, .planet1D, .GeoPoints
using Colors



In [ ]:
using Statistics

In [ ]:
using GLMakie
GLMakie.activate!()
Makie.inline!()

In [ ]:
p0 = GeoPoint(35.363602,138.726379) # summit of Mt Fuji


In [ ]:


Δx = 300.0 # in metre
Δy = 300.0
Δz = 300.0

altMax = 4.e3 # in metre
altMin = -10.e0 # in metre

horizontalDepth = 20.e3 

In [ ]:
boxGrids3D=constructLocalBox(p0,Δx,Δy,Δz,-horizontalDepth,horizontalDepth,-horizontalDepth,horizontalDepth,altMin,altMax)

In [ ]:
earthquakePoints = [GeoPoint(35.35,138.71),GeoPoint(35.36,138.72)]
stationPoints    = [GeoPoint(35.351,138.712),GeoPoint(35.361,138.721,alt=-3.e3)]  # array of GeoPoint


In [ ]:

extraPointSets = [
    GeoPointSet("earthquakes", earthquakePoints; color=:red, marker=:star5),
    GeoPointSet("stations", stationPoints; color=:blue, marker=:utriangle),
]

In [ ]:
eqLocal = [
    p_ECEF_to_local(p.ecef, p0.ecef, boxGrids3D.rotationMatrix)
    for p in earthquakePoints
]

staLocal = [
    p_ECEF_to_local(p.ecef, p0.ecef, boxGrids3D.rotationMatrix)
    for p in stationPoints
]

In [ ]:
eqLocal = GeoPoints_to_local(earthquakePoints, boxGrids3D)
staLocal = GeoPoints_to_local(stationPoints, boxGrids3D)

In [ ]:
seismicModel3D=lazyProduceOrLoad("seismicModel3D_Fuji",getParamsAndTopo,boxGrids3D.allGridsInGeoPoints,boxGrids3D.effectiveRadii,0.5)

In [ ]:
Nx3D,Ny3D,Nz3D=boxGrids3D.Nx, boxGrids3D.Ny, boxGrids3D.Nz

In [ ]:
boxGrids3D.allGridsInGeoPoints[1,1,1]

In [ ]:


x = [p.xyz[1] for p in boxGrids3D.allGridsInCartesian[:,1,1]]*1.e-3
y = [p.xyz[2] for p in boxGrids3D.allGridsInCartesian[1,:,1]]*1.e-3
z = [p.xyz[3] for p in boxGrids3D.allGridsInCartesian[1,1,:]]*1.e-3


ρ = seismicModel3D.ρ

sx = max(1, ceil(Int, size(ρ, 1) / 512))
sy = max(1, ceil(Int, size(ρ, 2) / 512))
sz = max(1, ceil(Int, size(ρ, 3) / 256))

ρ_plot = ρ[1:sx:end, 1:sy:end, 1:sz:end]
xg = x[1:sx:end]
yg = y[1:sy:end]
zg = z[1:sz:end]

println("original size = ", size(ρ))
println("plot size = ", size(ρ_plot))
println("strides = ", (sx, sy, sz))


In [ ]:




f = Figure()
ax = Axis3(f[1, 1])

volume!(
    ax,
    xg[1] .. xg[end],
    yg[1] .. yg[end],
    zg[1] .. zg[end],
    ρ_plot;
    algorithm = :absorption,
    colormap = :viridis,
)
f

# finding vertical gradient of Vpv

In [ ]:
Vp = seismicModel3D.Vpv

# Vertical Vp gradient, same size as Vp
dVp_dz = similar(Vp)

for k in 2:length(z)-1
    dVp_dz[:, :, k] .= (Vp[:, :, k+1] .- Vp[:, :, k-1]) ./ (z[k+1] - z[k-1])
end

dVp_dz[:, :, 1] .= (Vp[:, :, 2] .- Vp[:, :, 1]) ./ (z[2] - z[1])
dVp_dz[:, :, end] .= (Vp[:, :, end] .- Vp[:, :, end-1]) ./ (z[end] - z[end-1])
Gg = dVp_dz[1:sx:end, 1:sy:end, 1:sz:end];

In [ ]:
valid = filter(isfinite, vec(abs.(Gg)))
thr = quantile(valid, 0.98)   # strongest 2%
println("threshold = ", thr)

In [ ]:
f = Figure()
ax = Axis3(f[1, 1];
    xlabel = "x [km]",
    ylabel = "y [km]",
    zlabel = "z [km]",
)
contour!(
    ax,
    xg[1] .. xg[end],
    yg[1] .. yg[end],
    zg[1] .. zg[end],
    Gg;
    levels = [thr],
    color = (:red, 0.45),
    transparency = true,
)
contour!(
    ax,
    xg[1] .. xg[end],
    yg[1] .. yg[end],
    zg[1] .. zg[end],
    Gg;
    levels = [-thr],
    color = (:blue, 0.45),
    transparency = true,
)
f

In [ ]:
zTopo = Matrix{Float64}(undef, length(x), length(y))
changeStrength = Matrix{Float64}(undef, length(x), length(y))

for i in eachindex(x), j in eachindex(y)
    col = abs.(dVp_dz[i, j, :])
    kmax = argmax(col)

    zTopo[i, j] = z[kmax]
    changeStrength[i, j] = col[kmax]
end

In [ ]:
s = 20  # try 10, 20, 50

x2 = x[1:s:end]
y2 = y[1:s:end]
z2 = zTopo[1:s:end, 1:s:end]
c2 = changeStrength[1:s:end, 1:s:end]

f = Figure()
ax = Axis3(f[1, 1])
f = Figure()
ax = Axis3(f[1, 1])

volume!(ax,
    x2,
    y2,
    z2,
    c2,
    algorithm = :absorption,   # optional, makes it nicer
    colormap = :viridis
)

In [ ]:
display(f)

In [ ]:
println("zTopo size = ", size(zTopo))
println("changeStrength min/max = ", extrema(changeStrength))
println("zTopo min/max = ", extrema(zTopo))
println("finite masked points = ", count(isfinite, zTopoMasked), " / ", length(zTopoMasked))

In [ ]:
scatter!(
    first.(eqLocal),
    getindex.(eqLocal, 2),
    getindex.(eqLocal, 3);
    label="earthquakes",
    marker=:star5,
    color=:red,
)

scatter!(
    first.(staLocal),
    getindex.(staLocal, 2),
    getindex.(staLocal, 3);
    label="stations",
    marker=:utriangle,
    color=:blue,
)